# AZURA AQUA — Fine-tuning DialoGPT-small

Fine-tune `microsoft/DialoGPT-small` sur 1000 conversations aquaculture.

**Runtime** : GPU T4 (~20 min) | **Modèle** : 124M params | **Dataset** : 1000 conversations FR

In [ ]:
!pip install -q transformers datasets torch accelerate optimum[onnxruntime]

## 1. Générer le dataset AZURA AQUA

In [ ]:
import csv
import json
import random
from pathlib import Path

random.seed(42)

MODULES = ["Estran", "Finance", "Achats"]
ESTRAN_PARCS = ["P2", "P3", "P4", "P5", "P6", "P7", "P8", "P2-2", "P7-ext"]
ESTRAN_ZONES = ["Sud", "Est", "Ouest"]
ESTRAN_PHASES = ["grossissement", "Prégro", "Échantillonnage"]
ESTRAN_RECOLTES = ["Échantillonnage", "Transfert", "Récolte commerciale"]
FINANCE_LABELS = ["Chiffre d'affaires", "Matières premières", "MOD", "Frais généraux", "EBITDA"]
ACHAT_CATEGORIES = ["PLOMBERIE", "ELECTRICITÉ", "PRESTATION", "CARBURANTS", "INFORMATIQUE"]
ACHAT_FOURNISSEURS = ["CACED SARL", "AGATRAVE", "AIR LIQUIDE", "AGRIROUES"]
ACHAT_DEMANDEURS = ["SAID BOUKHLIK", "MAROUANE AITTOUGHA", "RIDA AYYI"]
ANOMALY_TYPES = ["Isolation Forest", "LOF", "One-Class SVM", "Z-Score"]
SEVERITY = ["Critique", "Majeure", "Mineure"]

def r_pct(): return f"{random.uniform(-30, 30):.1f}%"
def r_val(): return f"{random.randint(100, 500000):,}".replace(",", " ")
def r_biomasse(): return f"{random.randint(200, 2000)} kg"
def r_taux(): return f"{random.uniform(20, 90):.1f}%"

convs = []

# Estran
for _ in range(330):
    parc = random.choice(ESTRAN_PARCS)
    zone = random.choice(ESTRAN_ZONES)
    templates = [
        (f"Données du parc {parc} ?", f"Parc {parc}, zone {zone}. Phase : {random.choice(ESTRAN_PHASES)}. Biomasse : {r_biomasse()}. Taux recapture : {r_taux()}."),
        (f"Taux de recapture {parc} ?", f"Taux moyen {parc} : {r_taux()} pour {random.choice(ESTRAN_RECOLTES)}."),
        (f"Anomalies parc {parc} ?", f"{random.randint(1, 5)} anomalies ({random.choice(SEVERITY)}) via {random.choice(ANOMALY_TYPES)}. Biomasse écart : {r_biomasse()}.")
    ]
    convs.append(random.choice(templates))

# Finance
for _ in range(200):
    label = random.choice(FINANCE_LABELS)
    templates = [
        (f"Statut {label} ?", f"{label} : Budget {r_val()} DH, Réalisé {r_val()} DH. Variance : {r_pct()}."),
        ("Résumé YTD Finance", f"YTD : Budget {r_val()} DH, Réalisé {r_val()} DH. Variance : {r_pct()}. Driver : {random.choice(FINANCE_LABELS)}."),
        ("Anomalies financières ?", f"{random.randint(1, 8)} anomalies via {random.choice(ANOMALY_TYPES)}. Poste : {label}, variance {r_pct()}.")
    ]
    convs.append(random.choice(templates))

# Achats
for _ in range(280):
    templates = [
        ("DA en cours ?", f"{random.randint(10, 80)} DA en cours, {random.randint(5, 30)} sans BC. Valeur : {r_val()} DH."),
        ("Statut BC ?", f"BC en cours : {random.randint(20, 100)}. Livrées : {random.randint(50, 200)}."),
        (f"KPI {random.choice(ACHAT_DEMANDEURS)} ?", f"{random.randint(5, 50)} DA créées. BC : {random.randint(3, 40)}. Taux DA→BC : {random.randint(60, 95)}%."),
        ("Capex vs Opex ?", f"Opex : {random.randint(200, 500)} lignes, {r_val()} DH. Capex : {random.randint(50, 150)} lignes, {r_val()} DH.")
    ]
    convs.append(random.choice(templates))

# Général
greetings = [
    ("Bonjour", "Bonjour ! Assistant AZURA AQUA. Estran, Finance ou Achats ?"),
    ("Aide", "Modules : Estran (production), Finance (budget), Achats (DA/BC)."),
    ("Comment importer ?", "Glissez un .xlsx dans la barre d'import. Max 20 Mo."),
    ("Merci", "De rien ! N'hésitez pas."),
]
for _ in range(190):
    convs.append(random.choice(greetings))

random.shuffle(convs)
convs = convs[:1000]
print(f"Generated {len(convs)} conversations")

# Save
Path('data').mkdir(exist_ok=True)
with open('data/azura_conversations.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['prompt', 'response'])
    w.writerows(convs)
print("Saved to data/azura_conversations.csv")

## 2. Charger le modèle et tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "microsoft/DialoGPT-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print(f"Model: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 3. Préparer le dataset

In [ ]:
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling

dataset = Dataset.from_csv('data/azura_conversations.csv')
print(f"Dataset: {len(dataset)} conversations")

MAX_LENGTH = 128

def tokenize_fn(examples):
    texts = [f"{p}{tokenizer.eos_token}{r}{tokenizer.eos_token}"
             for p, r in zip(examples['prompt'], examples['response'])]
    tok = tokenizer(texts, truncation=True, padding='max_length', max_length=MAX_LENGTH)
    tok['labels'] = [[-100 if t == tokenizer.pad_token_id else t for t in ids] for ids in tok['input_ids']]
    return tok

tokenized = dataset.map(tokenize_fn, batched=True, batch_size=64, remove_columns=dataset.column_names)
split = tokenized.train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(split['train'])}, Eval: {len(split['test'])}")

## 4. Fine-tuning (~20 min sur T4)

In [ ]:
import torch
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir='./azura-dialogpt',
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=42,
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=collator,
)

trainer.train()

## 5. Évaluation + Test

In [ ]:
results = trainer.evaluate()
print(f"Eval loss: {results['eval_loss']:.4f}")
print(f"Perplexity: {torch.exp(torch.tensor(results['eval_loss'])):.2f}")

# Save
trainer.save_model('./azura-dialogpt')
tokenizer.save_pretrained('./azura-dialogpt')
print("Model saved to ./azura-dialogpt")

In [ ]:
# Quick test
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

test_prompts = [
    "Bonjour",
    "Combien de DA sont en cours ?",
    "Taux de recapture parc P7 ?",
    "Résumé YTD Finance",
    "Capex vs Opex ?",
]

for prompt in test_prompts:
    ids = tokenizer.encode(prompt + tokenizer.eos_token, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=80, do_sample=True, temperature=0.7, top_k=50, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    reply = tokenizer.decode(out[:, ids.shape[-1]:][0], skip_special_tokens=True)
    print(f"Q: {prompt}")
    print(f"A: {reply}\n")

## 6. Export ONNX + Push Hub (optionnel)

In [ ]:
from optimum.onnxruntime import ORTModelForCausalLM

# Export ONNX
ort_model = ORTModelForCausalLM.from_pretrained('./azura-dialogpt', export=True)
ort_model.save_pretrained('./azura-dialogpt-onnx')
tokenizer.save_pretrained('./azura-dialogpt-onnx')
print("ONNX model saved to ./azura-dialogpt-onnx")

In [ ]:
# Optional: Push to HuggingFace Hub
# from huggingface_hub import notebook_login
# notebook_login()
# ort_model.push_to_hub('YOUR_USERNAME/azura-dialogpt')
# tokenizer.push_to_hub('YOUR_USERNAME/azura-dialogpt')
# print("Pushed to Hub!")